In [0]:
# apache web server logs from elastic search
LOG_FILE_PATH = "/Volumes/workspace/default/raw/apache_logs.txt"

LOG_OUTPUT_CSV_PATH = (
    "/Volumes/workspace/default/bronze/elastic-apache-logs-csv"
)

LOG_OUTPUT_PARQUET_PATH = (
    "/Volumes/workspace/default/bronze/elastic-apache-logs-parquet"
)


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
import re
  
#   Read raw log lines from Volume
# unstructured input file
raw_df = spark.read.text(
    LOG_FILE_PATH
)

raw_df.printSchema()

print("schema ", raw_df.schema)
print("count ", raw_df.count())
print("first ", raw_df.first())
 

root
 |-- value: string (nullable = true)

schema  StructType([StructField('value', StringType(), True)])
count  10000
first  Row(value='83.149.9.216 - - [17/May/2015:10:05:03 +0000] "GET /presentations/logstash-monitorama-2013/images/kibana-search.png HTTP/1.1" 200 203023 "http://semicomplete.com/presentations/logstash-monitorama-2013/" "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_9_1) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/32.0.1700.77 Safari/537.36"')


In [0]:

# Apache Log Parser
# Python regular expression to extract data from log file, line by line
# return regex matches in tuple
apache_pattern = re.compile(
    r'(\S+) (\S+) (\S+) \[([^\]]+)\] '
    r'"(\S+) ([^"]*?) (\S+)" '
    r'(\d{3}) (\S+) '
    r'"([^"]*)" '
    r'"([^"]*)"'
)


In [0]:
# schema for structured dataframe with field, we still not yet extracted the data

# UDF Return Schema
 
parsed_schema = StructType([
    StructField("ip", StringType(), True),
    StructField("client_ident", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("timestamp", StringType(), True),
    StructField("method", StringType(), True),
    StructField("url", StringType(), True),
    StructField("protocol", StringType(), True),
    StructField("status_code", IntegerType(), True),
    StructField("bytes_sent", LongType(), True),
    StructField("referrer", StringType(), True),
    StructField("user_agent", StringType(), True)
])


In [0]:

# Parsing Function, known as UDF in DataFrame, User Defined Function
# we will discuss UDF internals, performance, how UDF executed later sessions
# UDF allows developer to write custom code

def parse_apache_log(line):

    m = apache_pattern.match(line)

    if not m:
        return (
            None, None, None, None,
            None, None, None, None,
            None, None, None
        )

    bytes_sent = m.group(9)

    return (
        m.group(1),                         # ip
        m.group(2),                         # client_ident
        m.group(3),                         # user_id
        m.group(4),                         # timestamp
        m.group(5),                         # method
        m.group(6),                         # url
        m.group(7),                         # protocol
        int(m.group(8)),                    # status
        None if bytes_sent == "-" else int(bytes_sent),
        m.group(10),                        # referrer
        m.group(11)                         # user_agent
    )

 #   Register UDF
 
parse_udf = F.udf(parse_apache_log, parsed_schema)


In [0]:
# Apply UDF

parsed_df = (
    raw_df
    .withColumn("parsed", parse_udf(F.col("value")))
)

# spark support nested schema, ie schema within schema, not that root is schema, parsed is another schema inside
parsed_df.printSchema()
 
parsed_df.show(5) #  truncate=False

root
 |-- value: string (nullable = true)
 |-- parsed: struct (nullable = true)
 |    |-- ip: string (nullable = true)
 |    |-- client_ident: string (nullable = true)
 |    |-- user_id: string (nullable = true)
 |    |-- timestamp: string (nullable = true)
 |    |-- method: string (nullable = true)
 |    |-- url: string (nullable = true)
 |    |-- protocol: string (nullable = true)
 |    |-- status_code: integer (nullable = true)
 |    |-- bytes_sent: long (nullable = true)
 |    |-- referrer: string (nullable = true)
 |    |-- user_agent: string (nullable = true)

+--------------------+--------------------+
|               value|              parsed|
+--------------------+--------------------+
|83.149.9.216 - - ...|{83.149.9.216, -,...|
|83.149.9.216 - - ...|{83.149.9.216, -,...|
|83.149.9.216 - - ...|{83.149.9.216, -,...|
|83.149.9.216 - - ...|{83.149.9.216, -,...|
|83.149.9.216 - - ...|{83.149.9.216, -,...|
+--------------------+--------------------+
only showing top 5 rows


In [0]:
# extract fields from parsed struct, we drop value here after
parsed_df = parsed_df.select("parsed.*")

parsed_df.printSchema()
# ensure all fields are extracted, if not fix them rightly 
parsed_df.show(5)

root
 |-- ip: string (nullable = true)
 |-- client_ident: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- method: string (nullable = true)
 |-- url: string (nullable = true)
 |-- protocol: string (nullable = true)
 |-- status_code: integer (nullable = true)
 |-- bytes_sent: long (nullable = true)
 |-- referrer: string (nullable = true)
 |-- user_agent: string (nullable = true)

+------------+------------+-------+--------------------+------+--------------------+--------+-----------+----------+--------------------+--------------------+
|          ip|client_ident|user_id|           timestamp|method|                 url|protocol|status_code|bytes_sent|            referrer|          user_agent|
+------------+------------+-------+--------------------+------+--------------------+--------+-----------+----------+--------------------+--------------------+
|83.149.9.216|           -|      -|17/May/2015:10:05...|   GET|/presentations/lo

In [0]:
# Optional: convert timestamp string
parsed_df = parsed_df.withColumn(
    "event_time",
    F.to_timestamp(
        F.col("timestamp"),
        "dd/MMM/yyyy:HH:mm:ss Z"
    )
)

parsed_df.printSchema()
 
parsed_df.show(5)

root
 |-- ip: string (nullable = true)
 |-- client_ident: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- method: string (nullable = true)
 |-- url: string (nullable = true)
 |-- protocol: string (nullable = true)
 |-- status_code: integer (nullable = true)
 |-- bytes_sent: long (nullable = true)
 |-- referrer: string (nullable = true)
 |-- user_agent: string (nullable = true)
 |-- event_time: timestamp (nullable = true)

+------------+------------+-------+--------------------+------+--------------------+--------+-----------+----------+--------------------+--------------------+-------------------+
|          ip|client_ident|user_id|           timestamp|method|                 url|protocol|status_code|bytes_sent|            referrer|          user_agent|         event_time|
+------------+------------+-------+--------------------+------+--------------------+--------+-----------+----------+--------------------+-----------------

In [0]:
# Save as CSV
# check with coalesce(1) to see the output, without that as well
(
    parsed_df
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", "true")
    .csv(LOG_OUTPUT_CSV_PATH)
)

print(f"Written to {LOG_OUTPUT_CSV_PATH}")

Written to /Volumes/workspace/default/bronze/elastic-apache-logs-csv


In [0]:
# Save as Parquet
# check with coalesce(1) to see the output, without that as well

(
    parsed_df
     .coalesce(1)
    .write
    .mode("overwrite")   
    .parquet(LOG_OUTPUT_PARQUET_PATH)
)

print(f"Written to {LOG_OUTPUT_PARQUET_PATH}")

Written to /Volumes/workspace/default/bronze/elastic-apache-logs-parquet
